<a href="https://colab.research.google.com/github/KanthiPhoosorn/CareMind/blob/feature%2Fpatient-staff-chatbots/CareMind_Patient_Custom_Local_Medical_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CareMind — Custom Citation-Based Medical Chatbot (No External LLMs)

This notebook demonstrates how to build a **fully local, custom medical chatbot system** for CareMind using ONLY:

- Your own hospital datasets
- Retrieval pipelines
- Rule-based reasoning
- Citation generation
- TF-IDF semantic matching
- No OpenAI
- No Gemini
- No Qwen
- No Ollama
- No hosted APIs

---

# Important Clarification

This notebook does **NOT** create a frontier large language model from scratch.

Instead, it builds:

```text
Medical Retrieval + Citation Engine
+ Workflow Intelligence
+ Rule-Based Clinical Responses
+ Search-Based Patient Assistant
```

This is the correct MVP architecture for CareMind.

---

# Why This Approach Matters

Training a true medical LLM from scratch requires:

- billions of tokens
- massive GPU clusters
- months of training
- extremely large budgets

For CareMind MVP, the correct architecture is:

```text
Structured Retrieval
+ Citation Engine
+ Medical Rules
+ Search Intelligence
```

This produces:
- safer outputs
- deterministic citations
- better PHI control
- easier hospital deployment


In [7]:

# Core imports

import os
import re
import glob
import pandas as pd

from dataclasses import dataclass
from typing import List, Dict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



# Step 1 — Load Hospital Data

This notebook loads the provided `data.zip` structure.

Expected layout:

```text
data/
  AN1/
    DoctorProgress_note.xlsx
    NURSE_Note.xlsx
    order (drug).xlsx
    order (lab).xlsx
    order (xray).xlsx
```


In [13]:
import os
import zipfile

zip_path = '/content/data.zip'
target_path = '/mnt/data/caremind_data'

# Create the directory if it doesn't exist
os.makedirs(target_path, exist_ok=True)

# Unzip the data
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(target_path)
    print(f'Successfully extracted {zip_path} to {target_path}')
else:
    print(f'Error: {zip_path} not found.')

Successfully extracted /content/data.zip to /mnt/data/caremind_data


In [14]:
# Check the extracted directory structure
for root, dirs, files in os.walk(target_path):
    level = root.replace(target_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files[:3]: # Show first 3 files
        print(f'{subindent}{f}')

caremind_data/
    data/
        AN2/
            AN2_NURSE_Note.xlsx
            AN2_order (lab).xlsx
            AN2_DoctorProgress_note.xlsx
        AN9/
            AN9_order (drug).xlsx
            AN9_NURSE_Note.xlsx
            AN9_DoctorProgress_note.xlsx
        AN4/
            AN4_order (lab).xlsx
            AN4_order (drug).xlsx
            AN4_NURSE_Note.xlsx
        AN8/
            AN8_NURSE_Note.xlsx
            AN8_order (drug).xlsx
            AN8_order (xray).xlsx
        AN6/
            AN6_NURSE_Note.xlsx
            AN6_order (xray).xlsx
            AN6_order (lab).xlsx
        AN3/
            AN3_order (xray).xlsx
            AN3_order (lab).xlsx
            AN3_NURSE_Note.xlsx
        AN7/
            AN7_DoctorProgress_note.xlsx
            AN7_NURSE_Note.xlsx
            AN7_order (xray).xlsx
        AN5/
            AN5_NURSE_Note.xlsx
            AN5_order (xray).xlsx
            AN5_DoctorProgress_note.xlsx
        AN1/
            AN1_order (xray).xlsx


In [15]:
# Find all Excel files in the correctly identified path
excel_files = glob.glob(
    os.path.join(target_path, "**", "*.xlsx"),
    recursive=True
)

print(f"Found {len(excel_files)} clinical data files.")

Found 48 clinical data files.


In [9]:

excel_files[:10]


[]


# Step 2 — Convert Clinical Files into Retrieval Chunks

Each row becomes a searchable citation chunk.

This is the foundation of citation-based AI.


In [16]:
@dataclass
class ClinicalChunk:
    chunk_id: str
    patient_an: str
    source_file: str
    row_index: int
    text: str

chunks: List[ClinicalChunk] = []
chunk_counter = 0

for file_path in excel_files:
    try:
        # Load the excel file
        df = pd.read_excel(file_path)

        # Extract the patient ID (AN) from the folder name
        patient_an = os.path.basename(os.path.dirname(file_path))
        source_file = os.path.basename(file_path)

        for idx, row in df.iterrows():
            # Filter out empty values and join row data into a single searchable string
            values = [str(v) for v in row.values if pd.notna(v)]
            combined_text = " | ".join(values).strip()

            if len(combined_text) < 5:
                continue

            chunks.append(ClinicalChunk(
                chunk_id=f"chunk_{chunk_counter}",
                patient_an=patient_an,
                source_file=source_file,
                row_index=idx,
                text=combined_text
            ))
            chunk_counter += 1
    except Exception as e:
        print(f"Failed to process: {file_path} due to {e}")

print(f"Successfully created {len(chunks)} searchable clinical chunks.")

Successfully created 9723 searchable clinical chunks.


In [17]:
# Rebuild the dataframe and vectorizer
chunk_df = pd.DataFrame([vars(x) for x in chunks])

if not chunk_df.empty:
    vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
    X = vectorizer.fit_transform(chunk_df["text"])
    print(f"Search index built with shape: {X.shape}")
else:
    print("Warning: No data chunks were created.")

Search index built with shape: (9723, 6746)



# Step 3 — Build Local Semantic Search

Instead of embeddings from external models, this notebook uses:

- TF-IDF vectors
- cosine similarity

Advantages:
- fully offline
- deterministic
- lightweight
- explainable
- CPU-friendly


In [18]:
# Re-initialize vectorizer on the full dataset to ensure consistency
vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
X = vectorizer.fit_transform(chunk_df["text"])
print(f"Vector matrix shape: {X.shape}")

Vector matrix shape: (9723, 6746)



# Step 4 — Local Retrieval Engine

This performs:
- semantic search
- patient filtering
- citation retrieval


In [19]:
def retrieve_chunks(query: str, patient_an: str, top_k: int = 5):
    scoped_df = chunk_df[chunk_df["patient_an"] == patient_an].copy()
    if scoped_df.empty:
        return pd.DataFrame()

    scoped_indices = scoped_df.index.tolist()
    scoped_matrix = X[scoped_indices]
    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(query_vector, scoped_matrix)[0]
    scoped_df["score"] = similarities
    scoped_df = scoped_df.sort_values(by="score", ascending=False)

    return scoped_df.head(top_k)

In [20]:
# Test retrieval
test_results = retrieve_chunks(query="fever and cough", patient_an="AN1")
test_results

,chunk_id,patient_an,source_file,row_index,text,score
7932,chunk_7932,AN1,AN1_order (xray).xlsx,0,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7933,chunk_7933,AN1,AN1_order (xray).xlsx,1,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7934,chunk_7934,AN1,AN1_order (xray).xlsx,2,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7935,chunk_7935,AN1,AN1_NURSE_Note.xlsx,0,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7936,chunk_7936,AN1,AN1_NURSE_Note.xlsx,1,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0


# Step 5 — Intent Detection

The chatbot should understand:
- symptom screening
- queue guidance
- medication questions
- appointment help
- FAQ

In [22]:
def detect_intent(query: str):
    q = query.lower()
    if any(x in q for x in ["wait", "queue", "appointment"]):
        return "workflow"
    if any(x in q for x in ["drug", "medicine", "medication"]):
        return "medication"
    if any(x in q for x in ["fever", "pain", "cough", "headache"]):
        return "symptom"
    return "faq"


# Step 6 — PHI Redaction

Before generation:
- remove AN
- remove HN
- remove identifiers


In [23]:
def redact_phi(text: str):
    text = re.sub(r"AN\d+", "[REDACTED_AN]", text)
    text = re.sub(r"HN\d+", "[REDACTED_HN]", text)
    return text


# Step 7 — Citation Builder

Every answer must include:
- source file
- row reference
- patient scope


In [24]:
def build_citations(results: pd.DataFrame):
    citations = []
    for _, row in results.iterrows():
        citation = f"{row['source_file']} (row {row['row_index']})"
        citations.append(citation)
    return citations


# Step 8 — Build a Custom Medical Chatbot

This is NOT an LLM.

It is:
- retrieval-based
- citation-grounded
- deterministic
- explainable
- hospital-safe


In [25]:
class CareMindMedicalChatbot:
    def answer(self, query: str, patient_an: str):
        intent = detect_intent(query)
        retrieved = retrieve_chunks(query=query, patient_an=patient_an)
        citations = build_citations(retrieved)
        if retrieved.empty:
            return {
                "intent": intent,
                "answer": "I could not find enough information to answer safely.",
                "citations": []
            }
        top_text = redact_phi(retrieved.iloc[0]["text"])
        if intent == "symptom":
            answer = f"Your records contain symptom-related information. Please consult hospital staff if symptoms worsen. Related record: {top_text[:250]}"
        elif intent == "medication":
            answer = "Medication-related records were found. Please verify medication usage with clinical staff."
        elif intent == "workflow":
            answer = "Workflow or appointment-related records were found."
        else:
            answer = "Relevant hospital records were found."
        return {
            "intent": intent,
            "answer": answer,
            "citations": citations
        }


# Step 9 — Test the Chatbot


In [26]:
# Final Test of the Chatbot
bot = CareMindMedicalChatbot()
response = bot.answer(query="fever and cough", patient_an="AN1")
response

{'intent': 'symptom',
 'answer': 'Your records contain symptom-related information. Please consult hospital staff if symptoms worsen. Related record: [REDACTED_AN] | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:02:23 | 20-DEC-2025 | 03:42:52 | E.C.G. (Electrocardiography)',
 'citations': ['AN1_order (xray).xlsx (row 0)',
  'AN1_order (xray).xlsx (row 1)',
  'AN1_order (xray).xlsx (row 2)',
  'AN1_NURSE_Note.xlsx (row 0)',
  'AN1_NURSE_Note.xlsx (row 1)']}

### Comprehensive Chatbot Testing
We will now test the system across different patient records and clinical intents.

In [27]:
test_scenarios = [
    {"query": "What medications are prescribed?", "an": "AN2"},
    {"query": "When is the next appointment or queue status?", "an": "AN10"},
    {"query": "Patient has a severe headache", "an": "AN3"},
    {"query": "Are there any lab results for glucose?", "an": "AN10"}
]

for scenario in test_scenarios:
    print(f"--- Testing Query: '{scenario['query']}' for Patient: {scenario['an']} ---")
    res = bot.answer(query=scenario['query'], patient_an=scenario['an'])
    display(res)
    print("\n")

--- Testing Query: 'What medications are prescribed?' for Patient: AN2 ---


{'intent': 'medication',
 'answer': 'Medication-related records were found. Please verify medication usage with clinical staff.',
 'citations': ['AN2_NURSE_Note.xlsx (row 0)',
  'AN2_NURSE_Note.xlsx (row 1)',
  'AN2_NURSE_Note.xlsx (row 2)',
  'AN2_NURSE_Note.xlsx (row 3)',
  'AN2_NURSE_Note.xlsx (row 4)']}



--- Testing Query: 'When is the next appointment or queue status?' for Patient: AN10 ---


{'intent': 'workflow',
 'answer': 'Workflow or appointment-related records were found.',
 'citations': ['AN10_order (lab).xlsx (row 275)',
  'AN10_DoctorProgress_note.xlsx (row 8)',
  'AN10_DoctorProgress_note.xlsx (row 39)',
  'AN10_DoctorProgress_note.xlsx (row 0)',
  'AN10_DoctorProgress_note.xlsx (row 2)']}



--- Testing Query: 'Patient has a severe headache' for Patient: AN3 ---


{'intent': 'symptom',
 'answer': 'Your records contain symptom-related information. Please consult hospital staff if symptoms worsen. Related record: [REDACTED_AN] | 2025-11-07 | 10:40:35 | 2025-12-16 | 16:20:29 | 12-DEC-2025 | 10:47 | PT Treatment\n- General exercise\n- Bed mobility training\n- Ambulation training\n\nPrecaution\n- Hypoxia\n- Falling\t | PT Note\n- General exercise UES and LES\n- Ambulation',
 'citations': ['AN3_NURSE_Note.xlsx (row 294)',
  'AN3_NURSE_Note.xlsx (row 385)',
  'AN3_NURSE_Note.xlsx (row 416)',
  'AN3_NURSE_Note.xlsx (row 273)',
  'AN3_NURSE_Note.xlsx (row 230)']}



--- Testing Query: 'Are there any lab results for glucose?' for Patient: AN10 ---


{'intent': 'faq',
 'answer': 'Relevant hospital records were found.',
 'citations': ['AN10_order (lab).xlsx (row 112)',
  'AN10_order (lab).xlsx (row 704)',
  'AN10_order (lab).xlsx (row 417)',
  'AN10_order (lab).xlsx (row 502)',
  'AN10_order (lab).xlsx (row 113)']}


# Step 10 — What Makes This Safer Than Generic LLMs

This architecture:
- never invents unsupported records
- only answers from retrieved evidence
- includes citations
- keeps PHI local
- avoids external APIs

---

# Step 11 — Future Upgrade Path

You can gradually evolve this into a true medical language model stack.

## Phase 1 (Current)
- TF-IDF retrieval
- deterministic responses
- citations
- rules

## Phase 2
Build your own embeddings:
- train SentencePiece tokenizer
- train clinical embeddings on Thai hospital data
- build vector search

## Phase 3
Train your own transformer:
- PyTorch
- decoder-only architecture
- clinical corpus pretraining
- instruction tuning
- retrieval grounding

## Phase 4
Fine-tune:
- Thai medical language
- queue workflows
- symptom triage
- hospital FAQ
- discharge instructions

---

# IMPORTANT

Do NOT start by training a giant LLM.

The correct order is:

```text
Data Cleaning
→ Retrieval
→ Citation Engine
→ Security
→ Evaluation
→ Embeddings
→ Small Transformer
→ Larger Models
```

For hospital systems:
- retrieval quality matters more than model size
- citations matter more than fluent text
- safety matters more than creativity
